In [ ]:
!pip install gradio transformers torch black autopep8 reportlab requests

In [ ]:
import gradio as gr
import ast
import re
import black
import autopep8

from transformers import pipeline

In [ ]:
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95

In [ ]:
print("Loading model...")

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
)

print("Model loaded successfully")

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
def detect_python_code(text):

    patterns = [
        r'def\s+\w+\s*\(',
        r'class\s+\w+',
        r'import\s+\w+',
        r'from\s+\w+\s+import',
        r'if\s+.*:',
        r'for\s+.*\s+in\s+',
        r'while\s+.*:'
    ]

    for p in patterns:
        if re.search(p, text):
            return True

    return False

In [ ]:
def check_syntax(code):

    try:
        ast.parse(code)
        return True, "No Syntax Error"

    except SyntaxError as e:
        return False, f"Syntax Error at line {e.lineno}: {e.msg}"

In [ ]:
def format_code(code):

    try:
        formatted = black.format_str(code, mode=black.FileMode())
        return formatted

    except:
        formatted = autopep8.fix_code(code)
        return formatted

In [ ]:
import ast
import black
import autopep8

# A basic knowledge base for Python theory (expandable)
PYTHON_THEORY = {
    "list comprehension": "A concise way to create lists. Syntax: [expr for item in iterable if condition]. Example: [x*2 for x in range(5) if x%2==0]",
    "dictionary comprehension": "Creates dictionaries. Syntax: {key_expr: value_expr for item in iterable if condition}. Example: {x:x**2 for x in range(5)}",
    "set comprehension": "Creates sets. Syntax: {expr for item in iterable if condition}. Example: {x for x in range(5)}",
    "tuple comprehension": "Python does not have tuple comprehensions. Use generator instead: (x for x in iterable)",
    "functions": "Functions are defined using 'def' keyword. They allow reusable code. Example: def add(a,b): return a+b",
    "classes": "Classes are blueprints for objects. Use 'class' keyword. Example: class Person: ...",
    "inheritance": "Inheritance allows one class to inherit properties/methods of another. Example: class Child(Parent): ...",
    "lambda": "Anonymous functions using lambda keyword. Example: lambda x: x**2",
    "decorators": "Functions that modify other functions. Example: @decorator above a function",
    "generators": "Functions that yield values using 'yield'. Useful for lazy evaluation",
    "exceptions": "Errors handled using try/except blocks. Example: try: x=1/0 except ZeroDivisionError: ...",
    "modules": "Files with Python code that can be imported using 'import'",
    "file handling": "Open/read/write files using open(), read(), write(), close() methods"
    # Add more concepts as needed
}

def fix_python_code_or_explain(input_text):
    # Detect if input is likely code
    code_keywords = ["def", "import", "for", "while", "if", "class", "=", "return", "try", "except"]
    if any(kw in input_text for kw in code_keywords):
        return fix_python_code(input_text)
    else:
        return explain_python_concept(input_text)

def fix_python_code(code):
    # Try syntax check
    try:
        ast.parse(code)
        formatted = black.format_str(code, mode=black.FileMode())
        return "✅ Code is correct. Formatted version:\n\n" + formatted
    except SyntaxError:
        # Attempt basic fixes
        lines = code.split("\n")
        fixed_lines = []

        for line in lines:
            stripped = line.rstrip()

            # Fix missing parentheses
            open_parens = stripped.count("(")
            close_parens = stripped.count(")")
            if open_parens > close_parens:
                stripped += ")" * (open_parens - close_parens)

            # Add colon for keywords
            keywords = ["def", "if", "for", "while", "class", "else", "elif", "try", "except"]
            for key in keywords:
                if stripped.strip().startswith(key) and not stripped.strip().endswith(":"):
                    stripped += ":"

            fixed_lines.append(stripped)

        fixed_code = "\n".join(fixed_lines)

        # Format code
        try:
            fixed_code = black.format_str(fixed_code, mode=black.FileMode())
        except:
            fixed_code = autopep8.fix_code(fixed_code)

        return "✅ Fixed Code:\n\n" + fixed_code

def explain_python_concept(question):
    question_lower = question.lower()
    for concept, explanation in PYTHON_THEORY.items():
        if concept in question_lower:
            return f"📖 Explanation for '{concept}':\n{explanation}"

    # If concept not in knowledge base, give a general answer
    return "❌ Sorry, concept not found in knowledge base. Try asking more specifically or expand the database."

# -------------------------
# Example Usage
# -------------------------
if __name__ == "__main__":
    # Example code input
    code_input = "def add(a,b\nreturn a+b"
    print(fix_python_code_or_explain(code_input))

    # Example theory input
    theory_input = "Explain lambda functions in Python"
    print("\n-----------------\n")
    print(fix_python_code_or_explain(theory_input))

✅ Fixed Code:

def add(a, b):


return a+b


-----------------

📖 Explanation for 'functions':
Functions are defined using 'def' keyword. They allow reusable code. Example: def add(a,b): return a+b


In [ ]:
conversation_history = []

def generate_response(user_message):

    if detect_python_code(user_message):

        result = fix_python_code(user_message)
        return result

    else:

        response = model_pipe(
            user_message,
            max_new_tokens=ChatbotConfig.DEFAULT_MAX_TOKENS,
            temperature=ChatbotConfig.DEFAULT_TEMP,
            top_p=ChatbotConfig.DEFAULT_TOP_P
        )

        return response[0]["generated_text"]

In [ ]:
interface = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=5, placeholder="Enter message or Python code"),
    outputs="text",
    title="Syntax Surgeon - AI Code Fixer",
    description="Paste Python code to fix syntax errors or ask questions"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f650d2ce30c68385e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
